#  Installation

In [ ]:
!pip install \
    langchain==0.3.27 \
    langchain-community==0.3.29 \
    langchain-core==0.3.76 \
    langchain-experimental==0.3.4 \
    langchain-google-community==2.0.10 \
    langchain-google-genai==2.1.12 \
    langchain-text-splitters==0.3.11 \
    langgraph==0.6.7 \
    langgraph-checkpoint==2.1.1 \
    langgraph-prebuilt==0.6.4 \
    langgraph-sdk==0.2.9 \
    elevenlabs \
    langchain_huggingface\
    ddgs\
    faiss-gpu-cu12\
    faiss-cpu\
    pandas\
    torch

    
    


# Imports


In [ ]:

# Data Handling
import pandas as pd
import torch

# LangChain Components
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.docstore.document import Document
from langchain.tools import tool

# Embeddings & Vector Stores
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# Models & Tools
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.tools import DuckDuckGoSearchRun

# LangGraph
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import MemorySaver


# Load data


In [ ]:
df = pd.read_csv("data\clean_wiki_movies.csv").sample(1000)


# Configuration 


In [ ]:
VECTOR_STORE_PATH = "data/movie_faiss_index"
EMBEDDING_MODEL_NAME = "BAAI/bge-m3"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"



# Build a simple vector database and implement a search functionality 


##  Document Creation 


In [ ]:

documents = []

for idx, row in df.iterrows():
    # Create rich document content
    content = f"""Title: {row['Title']}

Plot: {row['Plot_Clean'][:5000]}

Cast: {row['Cast']}

Director: {row['Director']}

Genre: {row['Genre']}"""


    # Create and append the document
    doc = Document(page_content=content)
    documents.append(doc)



##  Load the Hugging Face embedding model


In [ ]:
model_kwargs = {'device': DEVICE}
encode_kwargs = {'normalize_embeddings': False, 'batch_size': 16}

embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL_NAME,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

## Build and Save the FAISS Vector Store

In [ ]:
# Create the FAISS index from the documents and save it to disk

vector_store = FAISS.from_documents(documents, embeddings)


# Save the vector store
vector_store.save_local(VECTOR_STORE_PATH)


In [ ]:
loaded_vector_store = FAISS.load_local(
    VECTOR_STORE_PATH,
    embeddings,
    allow_dangerous_deserialization=True
)


## Search functions

In [ ]:

def search_movies(query, vector_store, k: int = 5) :
    results = vector_store.similarity_search_with_score(query, k=k)

    search_results = []
    for doc, score in results:
        search_results.append({
            "metadata": doc.metadata,
            "content": doc.page_content,
            "score": score
        })

    return search_results

## Test serach using Rag

In [ ]:
query = "a movie about dreams and the mind"
k = 3  # Number of results to return
res = search_movies(query, loaded_vector_store, k)

In [ ]:
print(res[0]["content"])

Title: Strawberry Shortcake: The Sweet Dreams Movie

Plot: after arranging a sleepover with her friends, strawberry and the rest travel to the land of dreams on a dreamboat that ginger snap has built, in order to stop the pie man from taking over their residence. 5 6

Cast: Sarah Heinke

Director: Karyn Hyden

Genre: animation


# Build a agent use (RAG) & internet search for Q&A


## Define Tools

In [ ]:
# Define the RAG tool as a function
@tool("RAG", return_direct=False)
def rag_tool(query, k= 2) :
    """Retrieve relevant knowledge base entries using vector similarity search."""
    try:
        results = vector_store.similarity_search(query, k=k)
        context = "\n\n".join([res.page_content for res in results])
        return f" Retrieved context:\n{context}"
    except Exception as e:
        return f"Error performing RAG search: {e}"



In [ ]:
# Initialize search tool
search = DuckDuckGoSearchRun()




## LLM

In [ ]:
# Initialize LLM
llm_api = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key="YOUR_GOOGLE_api",
)

## Agent Core

### Prompt

In [ ]:
# Define the system prompt
prompt = """You are a helpful assistant with access to a knowledge base and web search.
When asked a question:
1. First check the knowledge base using the RAG tool
2. If the knowledge base doesn't have sufficient information, use the duckduckgo_search
3. Combine information from both sources to provide a comprehensive answer
4. Always cite your sources when possible"""

## Memory

In [ ]:

# Create a single instance of the checkpointer to maintain state across requests
memory_checkpointer = MemorySaver()

## Initialize agent 

In [ ]:
agent = create_react_agent(
    model=llm_api,
    tools=[rag_tool, search],
    prompt=prompt,
    checkpointer=memory_checkpointer
)

## Test

In [ ]:
# Example usage of the agent
question = "movie for Director Richard Brooks"
thread_id = "thread_1"


response = agent.invoke(
    {"messages": [("human", question)]},
    config={"configurable": {"thread_id": thread_id}}
)

print(response["messages"][-1].content)

Richard Brooks directed the film *Wrong Is Right* (1982), a drama starring Sean Connery and Katharine Ross. The plot involves a globe-trotting reporter, Patrick Hale, who becomes entangled in a complex web of political intrigue, terrorism, and international espionage after interviewing King Ibn Awad. Hale attempts to uncover the truth as Awad threatens to detonate two suitcase nukes in Israel and the United States unless the U.S. President resigns.


In [ ]:
for i in response["messages"]:
  i.pretty_print()

================================ Human Message =================================

movie for Director Richard Brooks
================================== Ai Message ==================================
Tool Calls:
  RAG (6a11d6fa-73eb-4df9-b1b8-209d967ce08f)
 Call ID: 6a11d6fa-73eb-4df9-b1b8-209d967ce08f
  Args:
    query: movies directed by Richard Brooks
================================= Tool Message =================================
Name: RAG

 Retrieved context:
Title: Wrong Is Right

Plot: in the near future, violence has become something of a national sport and television news has fallen to tabloid depths. patrick hale, a globe-trotting reporter with access to a staggering array of world leaders, has ventured to the arab country of hegreb to interview his old acquaintance, king ibn awad. awad has learned that the president of the united states may have issued orders for his removal as a result, awad is apparently making arrangements to deliver two suitcase nukes to a terrorist, with t